# Lab 02: Multiple and Logistic regression

Author: **N.J. de Winter** (*n.j.de.winter@vu.nl*)<br>
Assitant Professor Vrije Universiteit Amsterdam<br>
Statistics and Data Analysis Course

## Learning goals:

* Practice your skills interpreting and modifying Python code 
* Understand and apply tools to test which *independent variables* in a larger dataset are relevant to predict the *dependent variable* (multiple regression
* Learn how to apply *regression* analysis to determine the *relationship* between multiple variables:
    * Multiple linear regression (straight line relationships between multiple variables)
    * Logistic regression (Logistic, or *sigmoid*-shaped relationships between multiple variables)

## Introduction
In this lab, we will get familiar with multiple regression and logistic regression using Python. Make sure you have completed Lab 01 first before starting on this one, because we will assume all the concepts in the first lab are familiar to you now. As a consequence, in this lab, there will be less "hand holding" and you will need to figure out more Python coding yourself. We will also assume from now on that the Jupyter format is familiar to you, although you are also still free to run the code in your own Spyder environment. If you ever get lost trying to get something done in Python, you can always go back to Lab 01 to check for examples about how to code things.

We will start again by loading the Python *packages* we need in this lab:

In [30]:
# Make sure our figures show up in Jupyter
%matplotlib inline
import numpy as np # The 'numpy' package contains some handy functions
import pandas as pd # The 'pandas' package helps us to import and manage data
import scipy.stats as stats # The 'scipy' package contains statistical formulas we need
import statsmodels.api as sm # The 'statsmodels' package contains the functions we need to run regression models
from sklearn import preprocessing # The 'sklearn' package contains functions we can use to pre-process our data
from matplotlib import pyplot as plt # The 'matplotlib' package contains tools need to plot our results
from mpl_toolkits.mplot3d import Axes3D # This is a cool package for making 3D-plots!

## Part 1: Multiple regression
With that our of the way, we can load the dataset we need to do our multiple regression. We will start with the dataset `Lab01b.csv`. You have worked with this dataset in the previous Lab, so it should be familiar to you.

__Exercise 1:__ Load the dataset here using the code box below. If you don't remember how to do this, look back to Lab01 and use the code you used there to load and inspect the data.

**Warning:** If you are working on this Lab (Lab02) in a different folder or environment on your computer than on Lab01 (which is a good idea!), you might need to copy the datafile you want to load into the folder you are working in for this Lab.

In [4]:
# Load the data for this assignment into Python and in the Jupyter environment.
df2 = pd.read_csv('Lab01b.csv')

# Inspect the first rows of the dataset to familiarize yourself again with this data
df2.head()

,Year,pCO2,pCH4,pN2O,SST
0,1850,284.001634,801.42,275.4,-0.30950
1,1851,287.128113,NaN,NaN,-0.19337
2,1852,NaN,NaN,NaN,-0.18339
3,1853,NaN,NaN,NaN,-0.21098
4,1854,288.054438,NaN,NaN,-0.26843


__Question 1:__ Which data is present in this dataset? Which are the independent variables? What is the dependent variable?

__Answer 1:__ The data contains concentrations of greenhouse gases CO2, CH4 and N2O and sea surface temperature anomaly (in degrees C relative to 1951-1980) since 1850. The greenhouse gas concentrations are independent variables. The temperature is the dependent variable

__Question 2:__ Based on the exercises in Lab01, what do you know about the correlations between these variables?

__Answer 2:__ SST is strongly positively correlated with all greenhouse gas concentrations.

In Lab01, we have assessed the correlations between variables one by one. This is feasible for relatively small datasets like this one, but quickly becomes tedious for larger datasets. Below you find a more efficient code to check correlations between all variables in a dataset. Try it!

In [11]:
A = np.corrcoef(df2, rowvar = False) # Create a matrix "A" which contains the Pearson's correlation coefficients between all variables.
print(A) # Print the resulting matrix A

[[ 1. nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]]


__Question 3:__ What went wrong here? (Hint: You were warned against this problem in Exercise 5 of Lab01)

__Answer 3:__ The dataset contains "NaN" values, which the "corrcoef" function could not handle.

We can solve this issue in two ways:
1. Modify our dataset to remove or mask the missing values
2. Use a different function which can deal with missing values

You can look up how to implement the first solution online if you are interested, but for now we will go with option 2 because it is faster. Run the code below to do so.

In [13]:
B = df2.corr() # Create correlation matrix
print(B) # Print the result

          Year      pCO2      pCH4      pN2O       SST
Year  1.000000  0.907769  0.975609  0.935363  0.812469
pCO2  0.907769  1.000000  0.928435  0.998395  0.925857
pCH4  0.975609  0.928435  1.000000  0.954541  0.902716
pN2O  0.935363  0.998395  0.954541  1.000000  0.956013
SST   0.812469  0.925857  0.902716  0.956013  1.000000


Notice that this is in fact the same function we used in Lab01, but without specifying the variables.

### Preparing data for multiple regression
In Lab01, we have focused on the relationship between CO2 and temperature, but now we will explore what the contribution of the other greenhouse gases might be. We will use a *multiple linear regression* to achive this. Running a multiple linear regression in Python is actually not that different from running a simple linear regression, but we need to prepare our data a little bit first:

In [99]:
# First we isolate the dependent variable (SST)
Y = df2[['SST']]

In [98]:
# Then we group the independent variables
X = df2[['pCO2', 'pCH4', 'pN2O']]

In [100]:
# Now, as with the polynomial regression, we need to add a constant (vector of ones) to tell Python that we want to include an intercept value in our regression
# The statsmodels package has a neat function to do this:
X = sm.add_constant(X)

__Exercise 2:__ Use the functions you have used before to inspect the newly created datasets `X` and `Y`:

In [86]:
X.head() # Inspect dataframe `X`

,const,pCO2,pCH4,pN2O
0,1.0,284.001634,801.42,275.4
1,1.0,287.128113,NaN,NaN
2,1.0,NaN,NaN,NaN
3,1.0,NaN,NaN,NaN
4,1.0,288.054438,NaN,NaN


In [87]:
Y.head() # Inspect dataframe `Y`

,SST
0,-0.30950
1,-0.19337
2,-0.18339
3,-0.21098
4,-0.26843


For our multiple regression, we will use a slightly different function from the `statsmodels` package than the one we used in Lab01. The syntax is almost the same as `smf.ols`:

In [66]:
multreg1 = sm.OLS(Y, X, missing = 'drop').fit()

__Question 4:__ What does the statement `missing = 'drop'` do in this function? (Hint: if you are unsure, you can always use the `help()` function to look it up!

__Answer 4:__ The statements causes the regression to ignore NaN values in the data

__Exercise 3:__ Use the code box below to print a summary of the multiple regression. (Hint: Use the same code you used in Lab01 to print summaries of simple linear regressions)

In [67]:
# Print a summary of the multiple linear regression
print(multreg1.summary())

                            OLS Regression Results                            
Dep. Variable:                    SST   R-squared:                       0.906
Model:                            OLS   Adj. R-squared:                  0.902
Method:                 Least Squares   F-statistic:                     190.6
Date:                Mon, 13 Mar 2023   Prob (F-statistic):           2.59e-30
Time:                        12:17:02   Log-Likelihood:                 56.708
No. Observations:                  63   AIC:                            -105.4
Df Residuals:                      59   BIC:                            -96.84
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.2933      0.169      1.739      0.0

__Question 5:__ Looking at the multiple regression overall, does the combination of independent variables accurately predict the value of the independent variable? Which regression metrics do you use to answer this question?

__Answer 5:__ Yes, the regression performed well, (adjusted) R2 > 0.9 and p-value of the regression is very low (<< 0.05)

__Question 6:__ Considering the contributions of the independent variables (greenhouse gas concentrations) separately, which variables significantly influence the dependent variable (temperature)? Which metrics do you use to answer this question?

__Answer 6:__ None of the independent variables seems to have a significant effect (p > 0.05) for all of them. The regression coefficients are also very low...

__Question 7:__ Your answer to __Question 6__ probably surprised you (especially after your analyses of CO2 vs SST in Lab01). Can you think of a problem in our analysis which caused this counter-intuitive result?

__Answer 7:__ Yes, the relationship between greenhouse gases and temperature is well established. The problem here is that the units of the variables are on very different scales. Also, the linear relationship is not ideal to describe the relationships between the variables.

Let's see if we can improve our multiple linear regression by *standardizing* the data so the values of the different variables are more comparable.

Remember from the Statistics part of the course: After standardization, our data will have a mean value of 0 and a standard deviation of 1. In other words, we are calculating z-scores for all variables

In [88]:
X_std = X # Copy the data from X into a new object
X_std[['pCO2', 'pCH4', 'pN2O']] = stats.zscore(X[['pCO2', 'pCH4', 'pN2O']], nan_policy = 'omit') # Normalize data in the new dataframe

# Repeat for Y
Y_std = Y
Y_std = stats.zscore(Y, nan_policy = 'omit')

__Exercise 4:__ Inspect the results of standardizing your variables, then repeay the multiple linear regression with the standardized data.

In [93]:
# Inspect standardized X data
X_std.head()

,const,pCO2,pCH4,pN2O
0,1.0,-1.206502,-1.77691,-1.579722
1,1.0,-1.120788,NaN,NaN
2,1.0,NaN,NaN,NaN
3,1.0,NaN,NaN,NaN
4,1.0,-1.095392,NaN,NaN


In [94]:
# Inspect standardized Y data
Y_std.head()

,SST
0,-0.737422
1,-0.361855
2,-0.329579
3,-0.418806
4,-0.604600


In [95]:
multreg2 = sm.OLS(Y_std, X_std, missing = 'drop').fit()
print(multreg2.summary())

                            OLS Regression Results                            
Dep. Variable:                    SST   R-squared:                       0.906
Model:                            OLS   Adj. R-squared:                  0.902
Method:                 Least Squares   F-statistic:                     190.6
Date:                Mon, 13 Mar 2023   Prob (F-statistic):           2.59e-30
Time:                        12:21:02   Log-Likelihood:                -17.237
No. Observations:                  63   AIC:                             42.47
Df Residuals:                      59   BIC:                             51.05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.2119      0.545      2.222      0.0

In [102]:
X2 = X.tail(44)
Y2 = Y.tail(44)
X2.head()

,const,pCO2,pCH4,pN2O
129,1.0,336.856667,1549.530,300.04
130,1.0,338.696667,1566.280,300.65
131,1.0,339.946667,1583.480,301.23
132,1.0,341.145000,1600.690,303.56
133,1.0,342.789167,1636.555,303.78


In [103]:
multreg3 = sm.OLS(Y2, X2, missing = 'drop').fit()
print(multreg3.summary())

                            OLS Regression Results                            
Dep. Variable:                    SST   R-squared:                       0.860
Model:                            OLS   Adj. R-squared:                  0.849
Method:                 Least Squares   F-statistic:                     79.97
Date:                Mon, 13 Mar 2023   Prob (F-statistic):           1.03e-16
Time:                        13:18:36   Log-Likelihood:                 52.227
No. Observations:                  43   AIC:                            -96.45
Df Residuals:                      39   BIC:                            -89.41
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -3.0717      4.530     -0.678      0.5

In [105]:
print(X2)

     const        pCO2         pCH4        pN2O
129    1.0  336.856667  1549.530000  300.040000
130    1.0  338.696667  1566.280000  300.650000
131    1.0  339.946667  1583.480000  301.230000
132    1.0  341.145000  1600.690000  303.560000
133    1.0  342.789167  1636.555000  303.780000
134    1.0  344.495000  1644.855000  304.020000
135    1.0  345.885833  1657.369167  304.540000
136    1.0  347.125833  1670.204167  305.370000
137    1.0  348.850833  1682.809167  305.550000
138    1.0  351.237500  1693.220000  306.490000
139    1.0  352.700000  1704.636667  307.480000
140    1.0  353.985833  1714.521667  308.780000
141    1.0  355.386667  1725.000833  309.570000
142    1.0  356.170833  1735.493333  310.000000
143    1.0  356.839167  1736.550833  310.250000
144    1.0  358.707500  1742.145833  310.980000
145    1.0  360.722500  1748.844167  311.780000
146    1.0  362.506667  1751.323333  312.810000
147    1.0  363.641667  1754.617500  313.530000
148    1.0  366.518333  1765.669167  314

In [132]:
# Load the data for this assignment into Python and in the Jupyter environment.
df3 = pd.read_csv('Lab02a.csv')

# Inspect the first rows of the dataset to familiarize yourself again with this data
df3.head()

,Year,Ozone hole area,Aerosols UK,Cropland area,Fossil fuel consumption,Freshwater use,SST
0,1979,100000.0,71.21,1.416061e+09,71546.138,3.110120e+12,0.14543
1,1980,1400000.0,66.20,NaN,70608.991,3.073440e+12,0.14203
2,1981,600000.0,63.69,1.421204e+09,69797.124,3.053970e+12,0.11512
3,1982,4800000.0,62.44,1.432903e+09,69040.456,3.067190e+12,0.09825
4,1983,7900000.0,61.82,1.438800e+09,69687.784,3.033540e+12,0.19460


In [133]:
# First we isolate the dependent variable (SST)
Y3 = df3[['SST']]

In [134]:
# Then we group the independent variables
X3 = df3[['Ozone hole area', 'Aerosols UK', 'Cropland area', 'Fossil fuel consumption', 'Freshwater use']]

In [135]:
# Now, as with the polynomial regression, we need to add a constant (vector of ones) to tell Python that we want to include an intercept value in our regression
# The statsmodels package has a neat function to do this:
X3 = sm.add_constant(X3)

In [136]:
Y3.head()

,SST
0,0.14543
1,0.14203
2,0.11512
3,0.09825
4,0.19460


In [137]:
multreg4 = sm.OLS(Y3, X3, missing = 'drop').fit()
print(multreg4.summary())

                            OLS Regression Results                            
Dep. Variable:                    SST   R-squared:                       0.761
Model:                            OLS   Adj. R-squared:                  0.707
Method:                 Least Squares   F-statistic:                     14.03
Date:                Mon, 13 Mar 2023   Prob (F-statistic):           3.20e-06
Time:                        13:45:07   Log-Likelihood:                 33.627
No. Observations:                  28   AIC:                            -55.25
Df Residuals:                      22   BIC:                            -47.26
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

In [126]:
print(len(Y3.index))

44
